# R3-U1: WavLM head under Protocol B
Select a Colab GPU runtime (T4 is enough; training itself runs on CPU). Sync the new code, `configs/r3_u1_wavlm_head_b.json`, the R2-U1 cache (`data/features/r2_u1_B`), the frozen manifests and the cohort shards to Drive first. Run cells in order. Reads training and validation embeddings only: no test partition, no Chatterbox. Outputs, checkpoints, the bundle and the ledger rows go to Drive; nothing is overwritten.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys, time
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Change if your project is elsewhere.
OWNER = 'Subhan'  # Person running this notebook; written to the ledger.
OUTPUT = ROOT / 'reports/r3/r3_u1_B'
BUNDLE = ROOT / 'bundles/r3_u1_wavlm_head_B'
MODEL_CACHE = '/content/hf_cache'  # Pinned WavLM is downloaded here and hash-verified; Colab-local, not synced.
assert (ROOT / 'scripts/r3_u1_wavlm_head.py').is_file(), 'Sync the updated project code to Drive'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, str(ROOT / 'src'))  # lets this kernel import echoproof, not only the subprocesses
print('Output:', OUTPUT, '\nBundle:', BUNDLE)

In [ ]:
import torch, transformers, numpy
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '| torch', torch.__version__,
      '| transformers', transformers.__version__, '| numpy', numpy.__version__)
assert torch.cuda.is_available(), 'Choose a GPU runtime (needed for the parity step)'
required = ['configs/r3_u1_wavlm_head_b.json', 'data/features/r2_u1_B/r2_u1_manifest.json', 'data/features/r2_u1_B/development_index.json',
            'data/manifests/cohort_v1.parquet', 'data/splits/protocol_B_v1.parquet', 'data/processed/chunks_v1.parquet',
            'reports/r1/split_report.json']
missing = [p for p in required if not (ROOT / p).is_file()]
assert not missing, f'Missing inputs: {missing}'
assert (ROOT / 'data/processed/cohort_v1').is_dir(), 'Cohort shards are needed for the parity step'
assert not OUTPUT.exists() or not any(OUTPUT.iterdir()), 'OUTPUT is not empty; choose a new folder (nothing is overwritten)'
assert not BUNDLE.exists(), 'BUNDLE already exists; choose a new folder'

## Fixture tests
Any failure or skip must be resolved before training.

In [ ]:
tests = ['tests/test_r3_u1_training.py', 'tests/test_r3_u1_alignment.py', 'tests/test_r3_u1_adapter.py']
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', *tests], text=True, capture_output=True)
print(check.stdout[-3000:], check.stderr[-2000:])
assert check.returncode == 0 and ' skipped' not in check.stdout, 'Resolve test failures or skips before training'

## Alignment, training, selection and export
`train` first proves the cache rows equal the Protocol B chunks and training recordings (writes `reports/r3/r3_u1_alignment_*.json`), trains `mean` and `mean_std` for 20 epochs each, freezes `selection.json` and then exports predictions for the three validation roles. Reading the cache can take several minutes per pooling.

In [ ]:
subprocess.run([sys.executable, 'scripts/r3_u1_wavlm_head.py', 'train', '--root', str(ROOT), '--owner', OWNER, '--device', 'cpu'], check=True)

## Direct-waveform parity
Scores selection-role chunks from the raw shard audio with the pinned WavLM (TF32 off) and the selected head, and compares them with the cached embeddings. A failure stops the notebook.

In [ ]:
subprocess.run([sys.executable, 'scripts/r3_u1_wavlm_head.py', 'parity', '--root', str(ROOT), '--owner', OWNER, '--device', 'cuda',
                '--model-cache', MODEL_CACHE], check=True)

## Package the verified inference bundle
Converts the pinned encoder weights once to safetensors (checked tensor by tensor) and writes the bundle. Uncalibrated: R3-N1 is still required.

In [ ]:
subprocess.run([sys.executable, 'scripts/r3_u1_wavlm_head.py', 'package', '--root', str(ROOT), '--owner', OWNER,
                '--model-cache', MODEL_CACHE], check=True)

## Verify the handoff and write the report
Checks recorded hashes, the completed ledger rows, the latest parity result and that the packaged bundle loads and scores a waveform. It does not replace independent review.

In [ ]:
import numpy as np
from echoproof.models.wavlm_adapter import WavLMBundleAdapter
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()
evidence = json.loads((OUTPUT / 'evidence.json').read_text(encoding='utf-8'))
assert evidence['status'] == 'validation_predictions_complete'
assert sha256(OUTPUT / 'selection.json') == evidence['selection_sha256']
assert all(sha256(OUTPUT / name) == digest for name, digest in evidence['output_sha256'].items())
selected = evidence['selection']['selected']
assert sha256(OUTPUT / f'run_{selected}' / 'best.pt') == evidence['selection']['checkpoint_sha256'][selected]
with (ROOT / 'reports/ledger.csv').open(newline='', encoding='utf-8') as f:
    rows = [r for r in csv.DictReader(f) if r['run_id'].startswith('r3_u1_') and r['status'] == 'completed']
assert len(rows) >= 2, 'expected two completed R3-U1 ledger rows'
parity_path = sorted((ROOT / 'reports/r3').glob('r3_u1_parity_*.json'))[-1]
parity = json.loads(parity_path.read_text(encoding='utf-8'))
assert parity['passed'] is True and parity['pooling'] == selected
adapter = WavLMBundleAdapter.load(BUNDLE, device='cuda')
probe = adapter.score_chunks([np.random.default_rng(0).normal(0, 0.05, 16000).astype(np.float32)])
assert probe.shape == (1,) and 0 <= probe[0] <= 1
print('selected:', selected, '| F1 by pooling:', evidence['selection']['selection_macro_f1'], '| parity max |dp|:', parity['max_abs_probability_difference'])
report = {'task': 'R3-U1', 'output': str(OUTPUT), 'bundle': str(BUNDLE), 'evidence_sha256': sha256(OUTPUT / 'evidence.json'),
          'parity_report': parity_path.name, 'parity_report_sha256': sha256(parity_path), 'completed_ledger_rows': rows,
          'gpu': torch.cuda.get_device_name(0), 'verification': 'hashes, ledger rows, parity and bundle load verified; independent review pending'}
report_path = ROOT / 'reports/r3' / ('r3_u1_colab_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '.json')
with report_path.open('x', encoding='utf-8') as f:
    json.dump(report, f, indent=2, sort_keys=True)
print('Report written:', report_path)